In [ ]:
# -*- coding: utf-8 -*-
#  Copyright 2026 United Kingdom Research and Innovation
#
#  Licensed under the Apache License, Version 2.0 (the "License");
#  you may not use this file except in compliance with the License.
#  You may obtain a copy of the License at
#
#      http://www.apache.org/licenses/LICENSE-2.0
#
#  Unless required by applicable law or agreed to in writing, software
#  distributed under the License is distributed on an "AS IS" BASIS,
#  WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
#  See the License for the specific language governing permissions and
#  limitations under the License.
#
#   Authored by:  Franck P. Vidal (UKRI-STFC)

# Example Contribution 

### Simulation of a radiograph of the Lungman phantom

This code simulates a  digital radiograph (DR) of the [Lungman anthropomorphic chest phantom](https://doi.org/10.1117/1.JMI.5.1.013504) (Kyoto Kagaku, Tokyo, Japan). It uses functions (from `lungman.py`).

### gVXR Version 2.1.0

In [ ]:
from gvxrPython3 import gvxr
print(gvxr.getVersionOfSimpleGVXR())


### The Dataset

This requires the Lungman dataset from  `lungman_data.zip` from https://zenodo.org/records/10782644/files/lungman_data.zip?download=1


The file [`lungman.py`](./lungman.py) will take care of this.

Import packages

In [ ]:
import numpy as np
from matplotlib import pyplot as plt
from gvxrPython3 import gvxr
from gvxrPython3.utils import plotScreenshot, interactPlotPowerLaw
from lungman import downloadLungman, extractLungmanSTL, loadLungmanMeshes

Download the data from [Zenodo](https://zenodo.org/records/10782644).

In [ ]:
zip_fname, lungman_path, mesh_path, CT_path = downloadLungman()

Extract the STL files from the ZIP file.

In [ ]:
stl_fname_set = extractLungmanSTL(zip_fname, lungman_path)

Create the simulation environment

In [ ]:
gvxr.createOpenGLContext()

We increase the size of the visualisation framebuffer to generate higher resolution screenshots. It does not affect the simulation.

In [ ]:
gvxr.setWindowSize(1000, 1000)

A sample is defined by its geometry (surface) and material composition. Note that you can transform (translate, scale
 and rotate) a sample.

In [ ]:
loadLungmanMeshes(mesh_path)

A detector is defined by its position, orientation, pixel resolution and the space between the centre of two consecutive pixels along its two axes. Here we also set a scintillator. To speed-up everything, we downscale the simulation by a factor of 4.

In [ ]:
gvxr.setDetectorPosition(0, -200, 0, "mm")
gvxr.setDetectorUpVector(0, 0, 1)
gvxr.setDetectorNumberOfPixels(512, 512)
gvxr.setDetectorPixelSize(1, 1, "mm")
gvxr.setScintillator("CsI", 600, "um")

print("Detector position:", gvxr.getDetectorPosition("mm"), "mm")
print("Detector up vector:", gvxr.getDetectorUpVector())
print("Detector number of pixels:", gvxr.getDetectorNumberOfPixels())
print("Pixel spacing:", gvxr.getDetectorPixelSpacing("mm"), "mm")


Plot the energy response of the detector.

In [ ]:
detector_response = np.array(gvxr.getEnergyResponse("keV"))

plt.figure(figsize= (20,10))
# plt.title("Detector response")
plt.plot(detector_response[:,0], detector_response[:,1])
plt.xlabel('Incident energy: E (in keV)')
plt.ylabel('Detector energy response: $\\delta$(E) (in keV)')

plt.tight_layout()

We must set the X-ray source's position and beam shape.


In [ ]:
gvxr.setSourcePosition(0, 1000, 0, "mm")
gvxr.useConeBeam()

print("Source position:", gvxr.getSourcePosition("mm"), "mm")
print("Source shape:", gvxr.getSourceShape())


We define here the number of photons and their kinetic energy.

In [ ]:
gvxr.setVoltage(80, "kV")
gvxr.setmAs(0.5)
gvxr.addInherentFilter("Al", 0.5, "mm") # Source
gvxr.addInherentFilter("C", 0.5, "mm") # Detector

Plot the beam spectrum computed with [xrayphysics](https://github.com/kylechampley/XrayPhysics).


In [ ]:
energy_bins = gvxr.getEnergyBins("keV")
photon_count = np.array(gvxr.getPhotonCountsPerCm2At1m(), dtype=np.single)

plt.figure(figsize= (20,10))
# plt.title("Beam spectrum")
plt.bar(energy_bins, photon_count, width=1)
plt.xlabel('Energy in keV')
plt.ylabel('Number of photons\nper cm2 at 1m')
plt.tight_layout()

Compute the corresponding X-ray image with and without noise.


In [ ]:
gvxr.enablePoissonNoise()
x_ray_image_with_noise = np.array(gvxr.computeXRayImage(), dtype=np.single) / gvxr.getTotalEnergyWithDetectorResponse()

In [ ]:
interactPlotPowerLaw(x_ray_image_with_noise, gamma=4.0)

Plot the corresponding screenshot.

In [ ]:
gvxr.setSceneRotationMatrix(
(
    0.6226841807365417, 0.1640920490026474, -0.7650728225708008, 0.0,
    0.7814211845397949, -0.18107721209526062, 0.5971522927284241, 0.0,
    -0.04054974392056465, -0.9696821570396423, -0.24098001420497894, 0.0,
    0.0, 0.0, 0.0, 1.0)
)
gvxr.setZoom(1700)
plotScreenshot()

Turn off the simulation.

In [ ]:
gvxr.destroy()